<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 1 — Python, NumPy, pandas & Matplotlib Refresher</div>
</div>

#### This notebook revises the Python skills introduced in PHYS281 that the rest of this module builds on. Nothing here should be genuinely new — if it is, that's worth flagging in the lab session so we can slow down.
***

## <span style="color:#B5121B">How to use this notebook</span>
This is deliberately paced as revision, not first exposure. Work through it actively — run every cell, change values, see what breaks. There's a short exercise after most sections, not just one block at the end — do them as you go rather than saving them all up.
***

## NumPy revision

In [ ]:
import numpy as np

# Arrays behave like vectors: operations apply elementwise, without an explicit loop
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([10.0, 20.0, 30.0, 40.0, 50.0])
print("x + y      =", x + y)
print("x * 2      =", x * 2)
print("mean(x)    =", x.mean())
print("std(x)     =", x.std())

**Vectorisation matters**: a NumPy array operation runs as compiled C code under the hood, not a Python loop. Compare a Python loop against the vectorised equivalent for the same calculation:

In [ ]:
import timeit

n = 2_000_000
data = np.random.default_rng(0).uniform(0, 1, size=n)

def loop_version():
    total = 0.0
    for v in data:
        total += v ** 2
    return total

def vectorised_version():
    return (data ** 2).sum()

# Take the best of a few repeats - a single timing is noisy, especially
# the very first call, which pays a one-off warm-up cost unrelated to the
# calculation itself.
t_loop = min(timeit.repeat(loop_version, number=1, repeat=3))
t_vec = min(timeit.repeat(vectorised_version, number=1, repeat=3))

print(f"Python loop : {t_loop:.4f} s   (result={loop_version():.4f})")
print(f"Vectorised  : {t_vec:.5f} s   (result={vectorised_version():.4f})")
print(f"Speed-up    : {t_loop / t_vec:.0f}x")

### From loops to linear algebra

That speed-up isn't a magic trick — it's because `(data ** 2).sum()` isn't just "a faster loop", it's a genuinely different way of describing the calculation. Watch what happens when we compute a dot product both ways:

In [ ]:
a = np.array([1.0, 2.0, 3.0, 4.0])
b = np.array([5.0, 6.0, 7.0, 8.0])

# As a loop - exactly what you'd write in plain Python
total = 0.0
for i in range(len(a)):
    total += a[i] * b[i]
print("loop result :", total)

# As linear algebra: a dot product
print("np.dot      :", np.dot(a, b))
print("@ operator  :", a @ b)

`for i in range(len(a)): total += a[i] * b[i]` and $\mathbf{a}\cdot\mathbf{b} = \sum_i a_i b_i$ are the *same statement* — one written as a Python loop, one written as linear algebra notation, one written as `a @ b`. NumPy's `@` operator (and `np.dot`) is doing exactly the loop above, just in compiled code that also knows how to exploit the hardware.

This extends past vectors. A matrix-vector multiply is a *loop of dot products* — one per output row:

In [ ]:
# A 3x4 matrix (e.g. three different weighted combinations of four inputs)
M = np.array([
    [1.0, 0.0, 2.0, 1.0],
    [0.0, 1.0, 0.0, 3.0],
    [2.0, 1.0, 1.0, 0.0],
])
v = np.array([1.0, 2.0, 3.0, 4.0])

# As nested loops
result_loop = np.zeros(M.shape[0])
for i in range(M.shape[0]):          # one dot product per row...
    row_total = 0.0
    for j in range(M.shape[1]):      # ...computed as a loop over columns
        row_total += M[i, j] * v[j]
    result_loop[i] = row_total
print("loop result   :", result_loop)

# As linear algebra
print("matrix result :", M @ v)

Same relationship: two nested loops (row, then column) *is* matrix-vector multiplication. When you replace a loop with an array operation, you are — usually without thinking about it — re-expressing your calculation as linear algebra. That's worth noticing now, because it stops being a background performance trick and becomes the main event in Week 6, where eigendecomposition and PCA use exactly this loop-as-matrix-operation idea as a first-class analysis tool, not just a speed-up.

> **Quick check:** without running it, what shape would `result_loop` be if `M` were 5×4 instead of 3×4? What about `v @ M.T` compared to `M @ v` — same numbers, different shape, or something else entirely? Check your answer by running it.

***
## pandas: loading and exploring data

`pandas` is built on top of NumPy and gives you labelled, tabular data — rows and columns, like a spreadsheet, but scriptable.

In [ ]:
import pandas as pd
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week1/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

df = pd.read_csv(data_path("pandas_practice_dataset.csv"))
df.head()

### Selecting, slicing and masking

In [ ]:
# A boolean mask selects rows matching a condition
high_calorie = df[df["Calories"] > 400]
high_calorie

In [ ]:
# Selecting specific columns
df[["Date", "Pulse", "Calories"]].head()

### Grouping and aggregating

> **If you've used SQL:** `.groupby()` is doing the same job as `GROUP BY`; the boolean mask above (`df[df["Calories"] > 400]`) is `WHERE`; and `.merge()`, which we'll use later in the module to combine datasets, is a `JOIN`. The syntax is different, but the underlying operation — group rows, filter rows, combine tables — is identical to what a database query does.

In [ ]:
# Mean calories burned, grouped by session duration
df.groupby("Duration")["Calories"].mean()

> **Quick check:** the `Maxpulse` column exists alongside `Pulse`. Compute, in one line, the mean *difference* between them (`Maxpulse - Pulse`) for each `Duration` group. Is the gap bigger for longer or shorter sessions?

***
## Matplotlib basics

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["Pulse"], df["Calories"], "o", color="dodgerblue")
ax.set_xlabel("Pulse (bpm)")
ax.set_ylabel("Calories burned")
ax.set_title("Calories vs. pulse")
plt.show()

In [ ]:
# A histogram, and an errorbar plot with synthetic uncertainties
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(df["Calories"], bins=8, color="mediumseagreen", edgecolor="black")
axes[0].set_xlabel("Calories burned")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of calories burned")

rng = np.random.default_rng(1)
y_err = rng.uniform(5, 20, size=len(df))
axes[1].errorbar(df["Duration"], df["Calories"], yerr=y_err, fmt="o", color="firebrick", capsize=3)
axes[1].set_xlabel("Duration (min)")
axes[1].set_ylabel("Calories burned")
axes[1].set_title("With (illustrative) uncertainties")

fig.tight_layout()
plt.show()

### Saving figures

Always save at a sensible resolution, with tight bounding boxes, and in a vector format (PDF) where possible — it stays sharp at any zoom level.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(df["Pulse"], df["Calories"], "o", color="dodgerblue")
ax.set_xlabel("Pulse (bpm)")
ax.set_ylabel("Calories burned")
fig.savefig("pulse_vs_calories.pdf", bbox_inches="tight", dpi=200)
print("Saved.")

***
## <span style="color:#FD9029">Exercises</span>

These get progressively less scaffolded — the last couple are deliberately open-ended.

1. Compute the mean **and** standard deviation of `Calories` for each `Duration` group (hint: `.agg(["mean", "std"])`), then produce a single, clearly labelled bar chart of average calories per duration group with error bars showing that standard deviation. Save it as a PDF.
2. Rewrite the matrix-vector multiply from the NumPy section above so it computes `M @ v` using only `np.dot` calls in a loop over rows (no `@`, no nested loop) — i.e. find the middle ground between the fully-looped version and the fully-vectorised one.
3. The dataset has a `Date` column stored as text. Convert it to a proper datetime type (`pd.to_datetime`), then plot `Calories` burned over time as a line, with the x-axis showing actual dates rather than row numbers.
4. **Open-ended:** using only tools introduced in this notebook, find and visualise one relationship in the data that hasn't already been shown above (e.g. does `Maxpulse` predict `Calories` better than `Pulse` does?). Produce one figure and one sentence stating what you found.
***

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis &middot; Lancaster University
</div>